<a href="https://www.kaggle.com/code/kristopherashmore/quantizing-qwen?scriptVersionId=312682088" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1: BASELINE INT8 QUANTIZATION VALIDATION (FIXED VERSION)
# ════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

device = "cpu"  # keep stable for Kaggle/P100

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("📦 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
).to(device).eval()

# ─────────────────────────────────────────────
# DATA (small but meaningful)
# ─────────────────────────────────────────────
def get_texts(n=32):
    try:
        ds = load_dataset("wikitext","wikitext-2-raw-v1",split="train")
        return [x["text"] for x in ds if len(x["text"].strip()) > 50][:n]
    except:
        return ["The quick brown fox jumps over the lazy dog."] * n

texts = get_texts()

# ─────────────────────────────────────────────
# CAPTURE ACTIVATIONS (FIXED)
# ─────────────────────────────────────────────
def capture_layer_output(model, layer_name):
    outputs = []

    def hook(module, inp, out):
        # FIX: reduce sequence dimension so shapes match
        out_reduced = out.detach().mean(dim=1)  # (batch, hidden_dim)
        outputs.append(out_reduced)

    handle = None
    for n, m in model.named_modules():
        if n == layer_name:
            handle = m.register_forward_hook(hook)
            break

    with torch.no_grad():
        for t in texts[:4]:
            inp = tokenizer(
                t,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=256
            ).to(device)

            model(**inp)

    if handle:
        handle.remove()

    return torch.cat(outputs, dim=0)

# ─────────────────────────────────────────────
# SIMPLE INT8 QUANT (RTN)
# ─────────────────────────────────────────────
def quantize_rtn(W):
    Wf = W.float()
    scale = Wf.abs().max(dim=1)[0] / 127.0
    scale = torch.clamp(scale, min=1e-8)

    Q = torch.round(Wf / scale.unsqueeze(1))
    Q = torch.clamp(Q, -127, 127)

    Wq = Q * scale.unsqueeze(1)
    return Wq

# ─────────────────────────────────────────────
# TEST EACH LAYER IN ISOLATION
# ─────────────────────────────────────────────
results = []

print("\n🔍 Evaluating layers one-by-one...\n")

for name, module in tqdm(list(model.named_modules())):
    if not isinstance(module, nn.Linear):
        continue

    if "lm_head" in name:
        continue

    # save original weights
    W_orig = module.weight.data.clone()

    # capture baseline output
    base_out = capture_layer_output(model, name)

    # apply quantization
    module.weight.data = quantize_rtn(W_orig)

    # capture quantized output
    quant_out = capture_layer_output(model, name)

    # restore original weights
    module.weight.data = W_orig

    # metrics
    mse = ((base_out - quant_out) ** 2).mean().item()

    cos = torch.nn.functional.cosine_similarity(
        base_out.flatten(),
        quant_out.flatten(),
        dim=0
    ).item()

    results.append((name, mse, cos))

# ─────────────────────────────────────────────
# ANALYSIS
# ─────────────────────────────────────────────
results_sorted = sorted(results, key=lambda x: x[1])

mse_vals = [r[1] for r in results]
cos_vals = [r[2] for r in results]

print("\n📊 GLOBAL STATS")
print(f"Mean MSE: {np.mean(mse_vals):.6f}")
print(f"Median MSE: {np.median(mse_vals):.6f}")
print(f"Min Cosine: {np.min(cos_vals):.4f}")
print(f"Mean Cosine: {np.mean(cos_vals):.4f}")

print("\n🟢 BEST 10 LAYERS")
for n, mse, cos in results_sorted[:10]:
    print(f"{n[:50]:50} | MSE={mse:.6f} | COS={cos:.4f}")

print("\n🔴 WORST 10 LAYERS")
for n, mse, cos in results_sorted[-10:]:
    print(f"{n[:50]:50} | MSE={mse:.6f} | COS={cos:.4f}")

📦 Loading model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


🔍 Evaluating layers one-by-one...



100%|██████████| 371/371 [1:23:05<00:00, 13.44s/it]


📊 GLOBAL STATS
Mean MSE: 0.000037
Median MSE: 0.000018
Min Cosine: 0.9997
Mean Cosine: 0.9999

🟢 BEST 10 LAYERS
model.layers.0.mlp.down_proj                       | MSE=0.000000 | COS=1.0000
model.layers.2.self_attn.o_proj                    | MSE=0.000000 | COS=1.0000
model.layers.4.self_attn.o_proj                    | MSE=0.000001 | COS=1.0000
model.layers.3.self_attn.o_proj                    | MSE=0.000001 | COS=1.0000
model.layers.0.self_attn.v_proj                    | MSE=0.000001 | COS=1.0000
model.layers.1.self_attn.v_proj                    | MSE=0.000001 | COS=1.0000
model.layers.3.mlp.down_proj                       | MSE=0.000001 | COS=1.0000
model.layers.13.mlp.down_proj                      | MSE=0.000001 | COS=0.9999
model.layers.4.mlp.down_proj                       | MSE=0.000002 | COS=1.0000
model.layers.14.mlp.down_proj                      | MSE=0.000002 | COS=0.9999

🔴 WORST 10 LAYERS
model.layers.27.self_attn.v_proj                   | MSE=0.000122 | COS=0.9999

In [3]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2: ADD ANALYTICAL CLIPPING (NO AWQ YET)
# ════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

device = "cpu"

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("📦 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
).to(device).eval()

fp_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
).to(device).eval()

# ─────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────
def get_texts(n=32):
    try:
        ds = load_dataset("wikitext","wikitext-2-raw-v1",split="train")
        return [x["text"] for x in ds if len(x["text"].strip()) > 50][:n]
    except:
        return ["The capital of France is Paris."] * n

texts = get_texts()

# ─────────────────────────────────────────────
# CLIPPED QUANT
# ─────────────────────────────────────────────
def quantize_clipped(W):
    Wf = W.float()

    std = Wf.std(dim=1)
    mx = Wf.abs().max(dim=1)[0]

    k = 2.8  # OmniQuant constant
    clip = torch.minimum(k * std, mx)
    clip = torch.clamp(clip, min=1e-5)

    scale = clip / 127.0

    Q = torch.round(Wf / scale.unsqueeze(1))
    Q = torch.clamp(Q, -127, 127)

    return Q * scale.unsqueeze(1)

# ─────────────────────────────────────────────
# APPLY FULL MODEL QUANT
# ─────────────────────────────────────────────
print("\n⚙️ Applying clipped quantization to entire model...")

for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        if "lm_head" in name:
            continue  # keep stable for now

        W = module.weight.data
        module.weight.data = quantize_clipped(W)

# ─────────────────────────────────────────────
# LOGIT EVALUATION
# ─────────────────────────────────────────────
@torch.no_grad()
def evaluate():
    prompt = "The capital of France is"
    inp = tokenizer(prompt, return_tensors="pt").to(device)

    a = fp_model(**inp).logits
    b = model(**inp).logits

    mse = ((a - b) ** 2).mean().item()
    cos = torch.nn.functional.cosine_similarity(
        a.flatten(), b.flatten(), dim=0
    ).item()

    print("\n🎯 LOGIT COMPARISON")
    print(f"MSE: {mse:.6f}")
    print(f"Cosine similarity: {cos:.4f}")

    # quick generation check
    out = model.generate(**inp, max_new_tokens=20)
    print("\n🧠 Generated:")
    print(tokenizer.decode(out[0]))

evaluate()

📦 Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


⚙️ Applying clipped quantization to entire model...

🎯 LOGIT COMPARISON
MSE: 14.249837
Cosine similarity: 0.4090

🧠 Generated:
The capital of France is located in the "Î" d tone location code. ﻿ #
Î
﻿|
"|The answer


In [4]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5: TRUE LAYER SENSITIVITY (LOGIT IMPACT BASED)
# ════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cpu"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("📦 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

fp_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).to(device).eval()

# ─────────────────────────────────────────────
# TEST PROMPT
# ─────────────────────────────────────────────
prompt = "The capital of France is"
inp = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    base_logits = fp_model(**inp).logits

# ─────────────────────────────────────────────
# QUANT METHODS
# ─────────────────────────────────────────────
def quant_clipped(W):
    Wf = W.float()
    std = Wf.std(dim=1)
    mx = Wf.abs().max(dim=1)[0]
    clip = torch.minimum(2.8 * std, mx).clamp(min=1e-5)
    scale = clip / 127.0
    Q = torch.round(Wf / scale.unsqueeze(1))
    Q = torch.clamp(Q, -127, 127)
    return Q * scale.unsqueeze(1)

# ─────────────────────────────────────────────
# TEST EACH LAYER
# ─────────────────────────────────────────────
layer_scores = []

print("\n🔍 Measuring true layer sensitivity...")

for name, module in fp_model.named_modules():
    if not isinstance(module, nn.Linear):
        continue

    # save original
    W_orig = module.weight.data.clone()

    # apply quant to THIS layer only
    module.weight.data = quant_clipped(W_orig)

    # forward
    with torch.no_grad():
        logits = fp_model(**inp).logits

    # compute damage
    mse = ((base_logits - logits) ** 2).mean().item()

    layer_scores.append((name, mse))

    # restore
    module.weight.data = W_orig

    print(f"{name} → impact: {mse:.6f}")

# sort by importance
layer_scores.sort(key=lambda x: x[1], reverse=True)

# top sensitive layers
k = int(len(layer_scores) * 0.15)
sensitive = set([x[0] for x in layer_scores[:k]])

print(f"\n🔥 Top {k} critical layers selected")

# ─────────────────────────────────────────────
# BUILD FINAL MODEL
# ─────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).to(device).eval()

for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue

    W = module.weight.data

    if name in sensitive or "lm_head" in name:
        # SAFE
        scale = W.abs().max(dim=1)[0] / 127.0
        scale = torch.clamp(scale, min=1e-8)
        Q = torch.round(W / scale.unsqueeze(1))
        Q = torch.clamp(Q, -127, 127)
        module.weight.data = Q * scale.unsqueeze(1)
    else:
        module.weight.data = quant_clipped(W)

# ─────────────────────────────────────────────
# FINAL EVAL
# ─────────────────────────────────────────────
with torch.no_grad():
    final_logits = model(**inp).logits

mse = ((base_logits - final_logits) ** 2).mean().item()
cos = torch.nn.functional.cosine_similarity(
    base_logits.flatten(), final_logits.flatten(), dim=0
).item()

print("\n🎯 FINAL RESULT")
print(f"MSE: {mse:.6f}")
print(f"Cosine similarity: {cos:.4f}")

out = model.generate(**inp, max_new_tokens=20)
print("\n🧠 Generated:")
print(tokenizer.decode(out[0]))

📦 Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🔍 Measuring true layer sensitivity...
model.layers.0.self_attn.q_proj → impact: 0.000563
model.layers.0.self_attn.k_proj → impact: 0.000140
model.layers.0.self_attn.v_proj → impact: 0.018270
model.layers.0.self_attn.o_proj → impact: 0.013622
model.layers.0.mlp.gate_proj → impact: 0.000963
model.layers.0.mlp.up_proj → impact: 0.000502
model.layers.0.mlp.down_proj → impact: 0.059352
model.layers.1.self_attn.q_proj → impact: 0.000198
model.layers.1.self_attn.k_proj → impact: 0.000132
model.layers.1.self_attn.v_proj → impact: 0.004287
model.layers.1.self_attn.o_proj → impact: 0.033650
model.layers.1.mlp.gate_proj → impact: 0.049540
model.layers.1.mlp.up_proj → impact: 0.014034
model.layers.1.mlp.down_proj → impact: 6.252412
model.layers.2.self_attn.q_proj → impact: 0.000334
model.layers.2.self_attn.k_proj → impact: 0.017570
model.layers.2.self_attn.v_proj → impact: 0.000097
model.layers.2.self_attn.o_proj → impact: 0.000226
model.layers.2.mlp.gate_proj → impact: 0.004708
model.layers.2.ml

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🎯 FINAL RESULT
MSE: 2.140620
Cosine similarity: 0.9075

🧠 Generated:
The capital of France is named after which famous king?
Paris
Charles I

King Charles I of England was known as the


In [5]:
# ════════════════════════════════════════════════════════════════════════
# STEP 6: PER-LAYER AUTO-TUNED CLIPPING (REPLACES AWQ)
# ════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cpu"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("📦 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

fp_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).to(device).eval()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).to(device).eval()

# ─────────────────────────────────────────────
# PROMPT
# ─────────────────────────────────────────────
prompt = "The capital of France is"
inp = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    base_logits = fp_model(**inp).logits

# ─────────────────────────────────────────────
# CLIP FUNCTION (WITH VARIABLE k)
# ─────────────────────────────────────────────
def quant_clip_k(W, k):
    Wf = W.float()
    std = Wf.std(dim=1)
    mx = Wf.abs().max(dim=1)[0]

    clip = torch.minimum(k * std, mx).clamp(min=1e-5)
    scale = clip / 127.0

    Q = torch.round(Wf / scale.unsqueeze(1))
    Q = torch.clamp(Q, -127, 127)

    return Q * scale.unsqueeze(1)

# ─────────────────────────────────────────────
# AUTO-TUNE EACH LAYER
# ─────────────────────────────────────────────
k_choices = [2.2, 2.5, 2.8, 3.2]
layer_best_k = {}

print("\n🔍 Auto-tuning per-layer clipping...")

for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue

    W_orig = module.weight.data.clone()

    best_k = None
    best_score = float("inf")

    for k in k_choices:
        module.weight.data = quant_clip_k(W_orig, k)

        with torch.no_grad():
            logits = model(**inp).logits

        mse = ((base_logits - logits) ** 2).mean().item()

        if mse < best_score:
            best_score = mse
            best_k = k

    layer_best_k[name] = best_k
    module.weight.data = W_orig

    print(f"{name} → best k: {best_k}, mse: {best_score:.6f}")

# ─────────────────────────────────────────────
# APPLY FINAL MODEL
# ─────────────────────────────────────────────
print("\n⚙️ Applying tuned quantization...")

for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue

    W = module.weight.data

    if "lm_head" in name:
        scale = W.abs().max(dim=1)[0] / 127.0
        scale = torch.clamp(scale, min=1e-8)
        Q = torch.round(W / scale.unsqueeze(1))
        Q = torch.clamp(Q, -127, 127)
        module.weight.data = Q * scale.unsqueeze(1)
    else:
        k = layer_best_k[name]
        module.weight.data = quant_clip_k(W, k)

# ─────────────────────────────────────────────
# FINAL EVAL
# ─────────────────────────────────────────────
with torch.no_grad():
    final_logits = model(**inp).logits

mse = ((base_logits - final_logits) ** 2).mean().item()
cos = torch.nn.functional.cosine_similarity(
    base_logits.flatten(), final_logits.flatten(), dim=0
).item()

print("\n🎯 FINAL RESULT")
print(f"MSE: {mse:.6f}")
print(f"Cosine similarity: {cos:.4f}")

out = model.generate(**inp, max_new_tokens=20)
print("\n🧠 Generated:")
print(tokenizer.decode(out[0]))

📦 Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🔍 Auto-tuning per-layer clipping...
model.layers.0.self_attn.q_proj → best k: 3.2, mse: 0.000257
model.layers.0.self_attn.k_proj → best k: 3.2, mse: 0.000086
model.layers.0.self_attn.v_proj → best k: 3.2, mse: 0.009696
model.layers.0.self_attn.o_proj → best k: 3.2, mse: 0.007618
model.layers.0.mlp.gate_proj → best k: 3.2, mse: 0.000603
model.layers.0.mlp.up_proj → best k: 3.2, mse: 0.000337
model.layers.0.mlp.down_proj → best k: 3.2, mse: 0.047096
model.layers.1.self_attn.q_proj → best k: 3.2, mse: 0.000070
model.layers.1.self_attn.k_proj → best k: 3.2, mse: 0.000059
model.layers.1.self_attn.v_proj → best k: 3.2, mse: 0.002573
model.layers.1.self_attn.o_proj → best k: 3.2, mse: 0.011859
model.layers.1.mlp.gate_proj → best k: 3.2, mse: 0.030132
model.layers.1.mlp.up_proj → best k: 3.2, mse: 0.008114
model.layers.1.mlp.down_proj → best k: 2.2, mse: 5.313690
model.layers.2.self_attn.q_proj → best k: 3.2, mse: 0.000192
model.layers.2.self_attn.k_proj → best k: 3.2, mse: 0.011943
model.lay

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3: QUANTIZED WEIGHT VALIDATION (DAIN‑DR RUNTIME SIMULATION)
# ═══════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import struct

# ──────────────────────────────────────────────────────────────────────────
# DAIN File Loader
# ──────────────────────────────────────────────────────────────────────────
def load_dain(path: Path):
    """
    Loads a .dain file and returns:
        w_int8  : np.ndarray (int8)   shape (out_features, in_features)
        scales  : np.ndarray (float32) shape (out_features,)
        awq_factor : np.ndarray (float32) or None shape (in_features,)
    """
    with open(path, 'rb') as f:
        magic = f.read(4)
        if magic != b'DAIN':
            raise ValueError(f"Invalid DAIN file: {path}")
        rows, cols, scale_len, awq_len = struct.unpack('>IIII', f.read(16))
        w_bytes = f.read(rows * cols)
        w_int8 = np.frombuffer(w_bytes, dtype=np.int8).reshape(rows, cols)
        scale_bytes = f.read(scale_len)
        scales = np.frombuffer(scale_bytes, dtype=np.float32)
        awq_factor = None
        if awq_len > 0:
            awq_bytes = f.read(awq_len)
            awq_factor = np.frombuffer(awq_bytes, dtype=np.float32)
    return w_int8, scales, awq_factor

# ──────────────────────────────────────────────────────────────────────────
# Dequantization & Layer Replacement
# ──────────────────────────────────────────────────────────────────────────
def dequantize_linear(w_int8, scales, awq_factor=None):
    """
    Converts int8 weights back to float32 tensor.
    If awq_factor is provided, the weights are assumed to be AWQ‑scaled,
    so we dequantize as: W = (W_int8 * scale) / awq_factor.
    (This matches the inference‑time activation scaling approach.)
    """
    w_fp32 = torch.from_numpy(w_int8).float() * torch.from_numpy(scales).unsqueeze(1)
    if awq_factor is not None:
        # AWQ factor should be applied to input activations.
        # For weight‑only comparison, we can fold it into the weight:
        w_fp32 = w_fp32 / torch.from_numpy(awq_factor).unsqueeze(0)
    return w_fp32

def replace_with_quantized(model, dain_dir: Path):
    """
    Replaces all nn.Linear layers in the model with dequantized weights
    loaded from .dain files.
    """
    dain_files = {p.stem: p for p in dain_dir.glob("*.dain")}
    replaced = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            safe_name = name.replace('.', '_')
            if safe_name in dain_files:
                w_int8, scales, awq_factor = load_dain(dain_files[safe_name])
                w_deq = dequantize_linear(w_int8, scales, awq_factor)
                # Check shape compatibility
                if w_deq.shape == module.weight.data.shape:
                    module.weight.data = w_deq.to(module.weight.data.dtype).to(module.weight.device)
                    replaced += 1
                else:
                    print(f"⚠️ Shape mismatch for {name}: {w_deq.shape} vs {module.weight.shape}")
    print(f"✅ Replaced {replaced} linear layers with quantized weights.")

# ──────────────────────────────────────────────────────────────────────────
# Validation Function
# ──────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def validate_model(original_model, quantized_model, tokenizer, prompt="The capital of France is"):
    """
    Compares logits from original and quantized models on a sample prompt.
    """
    device = next(original_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    original_model.eval()
    quantized_model.eval()
    
    out_orig = original_model(**inputs).logits
    out_quant = quantized_model(**inputs).logits
    
    mse = torch.nn.functional.mse_loss(out_orig, out_quant).item()
    cos_sim = torch.nn.functional.cosine_similarity(
        out_orig.flatten().unsqueeze(0), out_quant.flatten().unsqueeze(0)
    ).item()
    
    print(f"\n📊 Validation Results on prompt: '{prompt}'")
    print(f"   Logits MSE: {mse:.6f}")
    print(f"   Cosine similarity: {cos_sim:.6f}")
    
    # Also check a generated sequence
    orig_ids = original_model.generate(**inputs, max_new_tokens=20, do_sample=False)
    quant_ids = quantized_model.generate(**inputs, max_new_tokens=20, do_sample=False)
    
    orig_text = tokenizer.decode(orig_ids[0], skip_special_tokens=True)
    quant_text = tokenizer.decode(quant_ids[0], skip_special_tokens=True)
    
    print(f"\n📝 Original generation:  {orig_text}")
    print(f"📝 Quantized generation: {quant_text}")
    
    if orig_text == quant_text:
        print("✅ Generated texts match exactly!")
    else:
        print("⚠️ Generated texts differ (may be acceptable if similarity is high).")
    
    return mse, cos_sim

# ──────────────────────────────────────────────────────────────────────────
# Main Execution
# ──────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Paths (same as Cell 2)
    CACHE_DIR = "/kaggle/working/huggingface_cache"
    DAIN_DIR = Path("/kaggle/working/dain_dr/tuned_weights")
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    
    print(f"📦 Loading original FP16 model: {MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
    model_fp16 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", cache_dir=CACHE_DIR
    )
    model_fp16.eval()
    
    # Clone the model and replace weights with quantized versions
    print("🔄 Replacing weights with quantized versions...")
    model_quant = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", cache_dir=CACHE_DIR
    )
    replace_with_quantized(model_quant, DAIN_DIR)
    
    # Validate
    validate_model(model_fp16, model_quant, tokenizer)
    
    print("\n✅ Validation complete.")

📦 Loading original FP16 model: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

🔄 Replacing weights with quantized versions...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Replaced 0 linear layers with quantized weights.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📊 Validation Results on prompt: 'The capital of France is'
   Logits MSE: 0.000000
   Cosine similarity: 0.999512

📝 Original generation:  The capital of France is Paris. The capital of Italy is Rome. What is the capital of Spain?
A) Madrid

📝 Quantized generation: The capital of France is Paris. The capital of Italy is Rome. What is the capital of Spain?
A) Madrid

✅ Generated texts match exactly!

✅ Validation complete.
